In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Locate the project configuration from the working directory.
current_dir = Path.cwd().resolve()
project_root = next(
    (
        directory
        for directory in [current_dir, *current_dir.parents]
        if (directory / "config.py").is_file()
        and (directory / "notebooks").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not locate the project configuration.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config import (
    PROJECT_ROOT,
    RAW_DIR,
    BRONZE_DIR,
    SILVER_DIR,
    GOLD_DIR,
    TRIP_FILE,
    ZONE_FILE,
)

In [7]:
# Use the same file paths as in the raw-data inspection notebook.
trips = pd.read_parquet(TRIP_FILE)
zones = pd.read_csv(ZONE_FILE)
required_trip_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "total_amount",
]

missing_columns = [
    column
    for column in required_trip_columns
    if column not in trips.columns
]

if missing_columns:
    raise ValueError(f"Missing required trip columns: {missing_columns}")

if "LocationID" not in zones.columns:
    raise ValueError("Zone lookup is missing LocationID")

if zones["LocationID"].isna().any():
    raise ValueError("Zone lookup contains missing LocationID values")

if zones["LocationID"].duplicated().any():
    raise ValueError("Zone lookup contains duplicate LocationID values")

In [8]:
# Preserve the original data and assign a source-row reference.
quality = trips.copy().reset_index(drop=True)
quality.insert(0, "source_row_number", range(len(quality)))

pickup = pd.to_datetime(
    quality["tpep_pickup_datetime"], errors="coerce"
)
dropoff = pd.to_datetime(
    quality["tpep_dropoff_datetime"], errors="coerce"
)

month_start = pd.Timestamp("2026-01-01")
month_end = pd.Timestamp("2026-02-01")

valid_zone_ids = set(zones["LocationID"])
flags = pd.DataFrame(index=quality.index)

# Check timestamps without overwriting the original columns.
flags["pickup_time_missing"] = quality["tpep_pickup_datetime"].isna()
flags["pickup_time_unparseable"] = (
    quality["tpep_pickup_datetime"].notna() & pickup.isna()
)
flags["pickup_outside_target_month"] = (
    pickup.notna()
    & ((pickup < month_start) | (pickup >= month_end))
)

flags["dropoff_time_missing"] = quality["tpep_dropoff_datetime"].isna()
flags["dropoff_time_unparseable"] = (
    quality["tpep_dropoff_datetime"].notna() & dropoff.isna()
)
flags["dropoff_not_after_pickup"] = (
    pickup.notna() & dropoff.notna() & (dropoff <= pickup)
)

# Check location identifiers.
for column, prefix in [
    ("PULocationID", "pickup"),
    ("DOLocationID", "dropoff"),
]:
    flags[f"{prefix}_zone_missing"] = quality[column].isna()
    flags[f"{prefix}_zone_not_in_lookup"] = (
        quality[column].notna()
        & ~quality[column].isin(valid_zone_ids)
    )

# Check distance and monetary values separately.
for column in ["trip_distance", "fare_amount", "total_amount"]:
    numeric_values = pd.to_numeric(quality[column], errors="coerce")

    flags[f"{column}_missing"] = quality[column].isna()
    flags[f"{column}_unparseable"] = (
        quality[column].notna() & numeric_values.isna()
    )
    flags[f"{column}_negative"] = numeric_values.lt(0)
    flags[f"{column}_zero"] = numeric_values.eq(0)

if "passenger_count" in quality.columns:
    flags["passenger_count_missing"] = quality["passenger_count"].isna()

flags = flags.fillna(False).astype(bool)

# Calculate duration for inspection only.
quality["trip_duration_minutes"] = (
    dropoff - pickup
).dt.total_seconds() / 60

quality["has_quality_issue"] = flags.any(axis=1)

# Preserve every applicable reason instead of only the first one.
quality["quality_reasons"] = flags.apply(
    lambda row: "|".join(row.index[row]),
    axis=1,
)


In [9]:
quality_summary = pd.DataFrame({
    "issue_count": flags.sum(),
    "issue_pct": flags.mean().mul(100).round(3),
}).sort_values("issue_count", ascending=False)

print(f"Total rows: {len(quality):,}")
print(f"Rows with at least one issue: {quality['has_quality_issue'].sum():,}")

display(quality_summary)

# Issue counts overlap because a row can have multiple issues.
display(
    quality[
        ["trip_distance", "fare_amount", "total_amount",
         "trip_duration_minutes"]
    ].describe(percentiles=[0.01, 0.5, 0.95, 0.99]).T
)

inspection_columns = [
    "source_row_number",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "trip_duration_minutes",
    "quality_reasons",
]

display(
    quality.loc[
        quality["has_quality_issue"],
        inspection_columns,
    ].head(20)
)

Total rows: 3,724,889
Rows with at least one issue: 1,202,711


,issue_count,issue_pct
passenger_count_missing,1088058,29.210
trip_distance_zero,125738,3.376
dropoff_not_after_pickup,45070,1.210
total_amount_negative,39984,1.073
fare_amount_negative,39463,1.059
fare_amount_zero,2082,0.056
total_amount_zero,433,0.012
pickup_outside_target_month,7,0.000
total_amount_unparseable,0,0.000
total_amount_missing,0,0.000


,count,mean,std,min,1%,50%,95%,99%,max
trip_distance,3724889.0,6.455647,648.885528,0.0,0.00,1.810000,12.45,19.540000,269097.48
fare_amount,3724889.0,20.804254,18.927007,-2555.2,-3.00,15.600000,58.50,81.400000,2555.20
total_amount,3724889.0,29.178526,22.585530,-2560.2,-7.75,23.050000,75.25,104.750000,2560.20
trip_duration_minutes,3724889.0,17.193160,25.178020,-11.7,0.00,13.366667,42.75,71.433333,7507.90


,source_row_number,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,total_amount,trip_duration_minutes,quality_reasons
14,14,2026-01-01 00:40:35,2026-01-01 00:45:12,161,229,0.50,0.0,0.00,4.616667,fare_amount_zero|total_amount_zero
16,16,2026-01-01 00:45:15,2026-01-01 00:45:15,142,68,1.54,28.9,41.58,0.000000,dropoff_not_after_pickup
74,74,2026-01-01 00:54:05,2026-01-01 00:56:36,143,143,0.63,-5.1,-10.10,2.516667,fare_amount_negative|total_amount_negative
118,118,2026-01-01 00:21:36,2026-01-01 00:22:52,239,43,0.00,3.0,8.00,1.266667,trip_distance_zero
126,126,2026-01-01 00:46:05,2026-01-01 00:46:05,43,233,1.79,18.4,24.15,0.000000,dropoff_not_after_pickup
248,248,2026-01-01 00:43:29,2026-01-01 01:09:26,170,79,1.92,-22.6,-28.35,25.950000,fare_amount_negative|total_amount_negative
251,251,2026-01-01 00:22:40,2026-01-01 00:22:40,140,236,1.24,8.6,16.32,0.000000,dropoff_not_after_pickup
252,252,2026-01-01 00:36:04,2026-01-01 00:36:04,236,162,2.00,11.4,17.15,0.000000,dropoff_not_after_pickup
253,253,2026-01-01 00:51:23,2026-01-01 00:51:23,229,233,0.83,6.5,12.25,0.000000,dropoff_not_after_pickup
287,287,2026-01-01 00:48:27,2026-01-01 00:48:27,68,141,3.83,27.5,39.90,0.000000,dropoff_not_after_pickup
